# 🎮 Esports Matchmaking System using Machine Learning

## 📌 Overview
This project develops a **data-driven esports matchmaking system** that creates fair and balanced matches between teams.

Instead of relying only on raw performance metrics, the system introduces a custom feature called **Tactical Intelligence** and uses **machine learning** to predict match outcomes.

---

## 🎯 Objectives
- Analyze team and player performance data  
- Engineer meaningful features (Tactical Intelligence)  
- Predict match outcomes using ML  
- Recommend balanced opponents  
- Compare random vs optimized matchmaking  
- Validate fairness using simulation  

---

## 🧠 Key Concepts

### 🔹 Tactical Intelligence
A derived feature combining:
- Consistency (based on KD difference)
- Efficiency (KD vs rating)
- Experience (total maps played)

---

### 🔹 Team Strength
A weighted combination of:
- Rating  
- KD  
- Tactical Intelligence  

---

### 🔹 Match Prediction
- Model: Logistic Regression  
- Input: Difference between two teams  
- Output: Win probability  

---

### 🔹 Matchmaking Strategy
- Select opponent with win probability closest to **50%**
- Ensures balanced and competitive matches  

---

## ⚙️ Technologies Used
- Python  
- Pandas, NumPy  
- Matplotlib / Plotly  
- Scikit-learn  

---

## 📊 Workflow
1. Data Loading & Cleaning  
2. Feature Engineering  
3. Team Strength Calculation  
4. Model Training  
5. Match Prediction  
6. Matchmaking Optimization  
7. Simulation & Visualization  

---

> 💡 Goal: Improve matchmaking fairness using machine learning and data-driven insights

In [309]:
# Loading Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import plotly.express as px

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

In [310]:
team_df = pd.read_csv("team_stats.csv")
player_df = pd.read_csv("player_stats.csv")

# Drop useless column
team_df = team_df.drop(columns=['Unnamed: 0'])
player_df = player_df.drop(columns=['Unnamed: 0'])

# Rename for clarity
team_df = team_df.rename(columns={"name": "team"})
player_df = player_df.rename(columns={"name": "player"})

In [311]:
# Important Columns

team_df = team_df[['team','rating','kd','kd_diff','total_maps']]
player_df = player_df[['player','teams','rating','kd','kd_diff']]

In [312]:
team_df.info()
team_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 361 entries, 0 to 360
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   team        361 non-null    str    
 1   rating      361 non-null    float64
 2   kd          361 non-null    float64
 3   kd_diff     361 non-null    int64  
 4   total_maps  361 non-null    int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 14.2 KB


,rating,kd,kd_diff,total_maps
count,361.000000,361.000000,361.000000,361.000000
mean,1.006011,1.010388,759.664820,376.833795
std,0.024622,0.054000,2088.176536,360.930634
min,0.930000,0.850000,-2381.000000,101.000000
25%,0.990000,0.970000,-442.000000,143.000000
50%,1.010000,1.010000,221.000000,224.000000
75%,1.020000,1.040000,1266.000000,447.000000
max,1.070000,1.200000,11745.000000,1967.000000


In [313]:
player_df.info()
player_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1869 entries, 0 to 1868
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   player   1869 non-null   str    
 1   teams    1869 non-null   str    
 2   rating   1869 non-null   float64
 3   kd       1869 non-null   float64
 4   kd_diff  1869 non-null   int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 73.1 KB


,rating,kd,kd_diff
count,1869.000000,1869.000000,1869.000000
mean,0.985757,0.996726,180.254147
std,0.075545,0.100515,991.956591
min,0.680000,0.630000,-6238.000000
25%,0.940000,0.930000,-269.000000
50%,0.990000,0.990000,-32.000000
75%,1.030000,1.060000,352.000000
max,1.280000,1.430000,8070.000000


In [314]:
player_df.isnull().sum()

player     0
teams      0
rating     0
kd         0
kd_diff    0
dtype: int64

In [315]:
team_df.isnull().sum()

team          0
rating        0
kd            0
kd_diff       0
total_maps    0
dtype: int64

In [316]:
# Consistency (lower kd_diff = more controlled play)
team_df['consistency'] = 1 / (1 + abs(team_df['kd_diff']))

# Efficiency (kd to rating value)
team_df['kd_efficiency'] = team_df['kd'] / team_df['rating']

# Experience (total maps by team by total max maps played)
team_df['experience'] = team_df['total_maps'] / team_df['total_maps'].max()

In [317]:
# Normalize
scaler = MinMaxScaler()
cols = ['consistency', 'kd_efficiency', 'experience']
team_df[cols] = scaler.fit_transform(team_df[cols])

In [318]:
# Tactical Intelligence
team_df['tactical_intelligence'] = (
    0.4 * team_df['consistency'] +
    0.3 * team_df['kd_efficiency'] +
    0.3 * team_df['experience']
)

# Player Strength
player_df['player_strength'] = (
    0.6 * player_df['rating'] +
    0.4 * player_df['kd']
)

# Team Strength
team_df['team_strength'] = (
    0.5 * team_df['rating'] +
    0.3 * team_df['kd'] +
    0.2 * team_df['tactical_intelligence']
)

In [319]:
# Finding the best opponent for a team

def find_best_match(team_name):
    best_team = None
    best_score = float('inf')

    for other in team_df['team']:
        if other == team_name:
            continue

        prob = predict_win(team_name, other)

        # distance from perfect fairness
        fairness = abs(prob - 0.5)

        if fairness < best_score:
            best_score = fairness
            best_team = other

    return best_team

In [320]:
# Recommending a team

def recommend_opponent(team_name):
    return find_best_match(team_name)

In [321]:
# Creating Pairs

matches = []

for i in range(len(team_df)):
    for j in range(i+1, len(team_df)):
        t1 = team_df.iloc[i]
        t2 = team_df.iloc[j]

        strength_diff = t1['team_strength'] - t2['team_strength']
        ti_diff = t1['tactical_intelligence'] - t2['tactical_intelligence']

        # Combine both
        score = 1.5 * strength_diff + 3 * ti_diff   

        # Convert to probability (sigmoid)
        prob = 1 / (1 + np.exp(-score))

        # Decide winner
        win = 1 if np.random.rand() < prob else 0

        matches.append([strength_diff, ti_diff, win])
        matches.append([-strength_diff, -ti_diff, 1 - win])

        matches.append([strength_diff, ti_diff, win])

match_df = pd.DataFrame(matches, columns=['strength_diff', 'ti_diff', 'win'])

In [322]:
# Training Model on the Pairs

X = match_df[['strength_diff', 'ti_diff']]
y = match_df['win']

model = LogisticRegression()
model.fit(X, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [323]:
# Predicting Who Wins

def predict_win(team1, team2):
    t1 = team_df[team_df['team'] == team1].iloc[0]
    t2 = team_df[team_df['team'] == team2].iloc[0]

    df_input = pd.DataFrame([{
        'strength_diff': t1['team_strength'] - t2['team_strength'],
        'ti_diff': t1['tactical_intelligence'] - t2['tactical_intelligence']
    }])

    return model.predict_proba(df_input)[0][1]

In [324]:
# Simulation running 100 Matches

def simulate_match(team1, team2, n=100):
    prob = predict_win(team1, team2)

    wins = sum(1 for _ in range(n) if random.random() < prob)

    return {
        "team1": team1,
        "team2": team2,
        "team1_wins": wins,
        "team2_wins": n - wins,
        "win_probability": prob
    }

In [325]:
# Visualizing the Strength of the teams

fig = px.histogram(
    team_df,
    x='team_strength',
    nbins=30,
    title='Team Strength Distribution',
)

fig.show()

In [326]:
diff = team_df['tactical_intelligence'] - team_df['team_strength']

fig = px.scatter(
    team_df,
    x='team_strength',
    y='tactical_intelligence',
    
    hover_name='team',
    
    size='total_maps',   
    
    color=diff,          
    
    color_continuous_scale='RdBu',
    
    title='Team Strength vs Tactical Intelligence (Balanced View)',
    
    labels={
        'team_strength': 'Raw Strength',
        'tactical_intelligence': 'Tactical Intelligence'
    }
)

fig.update_layout(template='plotly_dark')

fig.show()

In [ ]:
# Plotting Team Comparison

def plot_match(team1, team2):
    t1 = team_df[team_df['team'] == team1].iloc[0]
    t2 = team_df[team_df['team'] == team2].iloc[0]

    df_plot = pd.DataFrame({
        'Metric': ['Strength', 'Tactical'],
        team1: [t1['team_strength'], t1['tactical_intelligence']],
        team2: [t2['team_strength'], t2['tactical_intelligence']]
    })

    df_melt = df_plot.melt(id_vars='Metric', var_name='Team', value_name='Value')

    fig = px.bar(
        df_melt,
        x='Metric',
        y='Value',
        color='Team',
        barmode='group',
        title=f"{team1} vs {team2} — Direct Comparison"
    )

    fig.show()

In [333]:
# -------- FINAL OUTPUT -------- #

team1 = team_df['team'].sample(1).values[0]
random_team = team_df[team_df['team'] != team1]['team'].sample(1).values[0]
best = find_best_match(team1)

print("🎮 MATCHMAKING ANALYSIS")
print("="*45)

print(f"\n🔹 Selected Team: {team1}")

# ================= RANDOM MATCH ================= #
prob_random = predict_win(team1, random_team)

print(f"\n🎲 Random Opponent: {random_team}")
print(f"Win Probability:")
print(f"{team1}: {round(prob_random*100,2)}% | {random_team}: {round((1-prob_random)*100,2)}%")

print("\n📊 Random Match Comparison:")
plot_match(team1, random_team)

print("\n🎲 Simulation (Random Match):")
sim_random = simulate_match(team1, random_team)
print(f"{team1}: {sim_random['team1_wins']} wins")
print(f"{random_team}: {sim_random['team2_wins']} wins")


# ================= BEST MATCH ================= #
prob_best = predict_win(team1, best)

print(f"\n🤝 Best Opponent: {best}")
print(f"Win Probability:")
print(f"{team1}: {round(prob_best*100,2)}% | {best}: {round((1-prob_best)*100,2)}%")

print("\n📊 Best Match Comparison:")
plot_match(team1, best)

print("\n🎲 Simulation (Best Match):")
sim_best = simulate_match(team1, best)
print(f"{team1}: {sim_best['team1_wins']} wins")
print(f"{best}: {sim_best['team2_wins']} wins")


# ================= FAIRNESS ================= #
fair_random = abs(prob_random - 0.5)
fair_best = abs(prob_best - 0.5)

print("\n⚖️ Fairness Comparison (closer to 0 = better):")
print(f"Random Match: {round(fair_random,3)}")
print(f"Best Match: {round(fair_best,3)}")



🎮 MATCHMAKING ANALYSIS

🔹 Selected Team: Astralis

🎲 Random Opponent: MAD Lions
Win Probability:
Astralis: 63.61% | MAD Lions: 36.39%

📊 Random Match Comparison:



🎲 Simulation (Random Match):
Astralis: 68 wins
MAD Lions: 32 wins

🤝 Best Opponent: forZe
Win Probability:
Astralis: 50.24% | forZe: 49.76%

📊 Best Match Comparison:



🎲 Simulation (Best Match):
Astralis: 49 wins
forZe: 51 wins

⚖️ Fairness Comparison (closer to 0 = better):
Random Match: 0.136
Best Match: 0.002


# ✅ Conclusion

In this project, we developed a **machine learning-based matchmaking system** that improves fairness in esports matches.

---

## 🔍 Key Outcomes
- Created a custom feature **Tactical Intelligence** to capture strategic gameplay  
- Built a **Logistic Regression model** to predict match outcomes  
- Designed a matchmaking system based on **balanced win probability**  
- Compared **random vs optimized matchmaking**  
- Validated results using **simulation**  

---

## ⚖️ Fairness Insight
- Random matchmaking often produces **unbalanced matches**  
- Optimized matchmaking produces **near 50-50 outcomes**  
- Fairness is measured using:  win_probability - 0.5 
- Lower value → more balanced match  

---

## 🚀 Final Conclusion
The system successfully demonstrates that:

> “Using machine learning and feature engineering can significantly improve matchmaking fairness compared to random pairing.”

---

## 🎯 Final Note
This project highlights how **data science + machine learning** can enhance competitive gaming experiences through intelligent matchmaking.
